# Milestone 5 – Workflow Design & Testing

This notebook documents the workflow design and testing for my two RenovAI scenarios from Milestone 3:

- **Scenario 1:** Accessible Bathroom Remodel (Aging in Place)
- **Scenario 2:** Small Patio Transformation (Renter-Friendly)

This notebook includes:
- A router workflow for classifying user intent
- A workflow for each scenario
- Prompt-based implementation of each workflow
- End-to-end testing using the Milestone 3 scenarios
- Evidence that outputs align with intended workflow behavior

All workflows are visualized using Mermaid.js flowcharts.


## Router Workflow – Classifying User Intent

RenovAI uses a lightweight router model to determine which workflow should handle a user’s request.

The router classifies user input into one of three categories:
- **Bathroom accessibility & safety remodels** → Scenario 1 workflow
- **Renter-friendly patio / outdoor design** → Scenario 2 workflow
- **Other or off-topic requests** → generic RenovAI refusal and redirection


```mermaid
flowchart TD
  A[User Message] --> B{Router LLM\nClassify Intent}
  B -->|Bathroom accessibility| S1[Scenario 1 Workflow\nAccessible Bathroom Remodel]
  B -->|Renter patio design| S2[Scenario 2 Workflow\nSmall Patio Transformation]
  B -->|Other / Off-topic| O[Generic RenovAI Response\nRefuse + Redirect]
```


## Scenario 1 Workflow – Accessible Bathroom Remodel

This workflow supports aging-in-place bathroom renovations by identifying safety risks, applying ADA-inspired guidelines, and generating three tiered renovation plans with cost estimates and a contractor-ready brief.


```mermaid
flowchart TD
  A1[Scenario 1 User Input\nMaria, 68, 7'x9', $15k] --> R1[Router LLM]
  R1 --> I1[Intake Prompt\nSummarize + ask clarifying questions]
  I1 --> B1[Bathroom Brief Builder\nGoals + constraints + safety risks]
  B1 --> G1[Option Generator\n3 accessibility tiers + cost ranges]
  G1 --> C1[Contractor Brief Generator]
  C1 --> K1[Budget & Constraint Check]
  K1 --> O1[Final Structured Output\nNext steps]
```


## Scenario 2 Workflow – Small Patio Transformation

This workflow designs renter-friendly outdoor spaces by respecting lease constraints, optimizing small layouts, and producing budget-aware design options with a shopping list and setup steps.


```mermaid
flowchart TD
  A2[Scenario 2 User Input\n6'x12', renter, $2k] --> R2[Router LLM]
  R2 --> I2[Intake Prompt\nSummarize + ask clarifying questions]
  I2 --> B2[Patio Brief Builder\nGoals + constraints]
  B2 --> G2[Design Generator\n3 renter-safe options]
  G2 --> S2[Shopping List Generator]
  S2 --> P2[Setup Steps Generator]
  P2 --> K2[Budget & Constraint Check]
  K2 --> O2[Final Structured Output\nNext steps]
```


## Implementation (Prompt-Based)

The workflows were implemented using **prompt chaining** rather than code.

Implementation steps:
1. A router prompt classifies user intent.
2. Based on routing, the appropriate scenario prompt from Milestone 3 is executed.
3. Each scenario prompt produces a complete structured response following the defined workflow steps.

No programmatic orchestration was used; instead, each workflow was executed sequentially using an LLM interface.


## Router Prompt

```text
You are the RenovAI router.

Classify the user request into ONE category:
- scenario_1_bathroom_accessibility
- scenario_2_renter_patio
- other

Return ONLY JSON:
{"route": "...", "reason": "..."}
```


## Scenario 1 Workflow Execution Evidence

### System Prompt (Scenario 1)
```text
SYSTEM: You are RenovAI, an accessibility-focused renovation assistant. Your goal is to help homeowners plan safe, ADA-informed bathroom remodels while staying within budget and timeline constraints.

IMPORTANT RULES (STRICT):
- Use ONLY the provided context data for ADA_guidelines and cost_estimates.
- Do NOT reference external websites, brands, contractors, or sources.
- Do NOT invent prices outside the provided cost_estimates.
- Follow the required response format EXACTLY.

Your tasks:
1) Restate the user’s bathroom size, layout, goals, and constraints.
2) Ask up to 3 follow-up questions if dimensions or mobility needs are unclear.
3) Identify safety and accessibility issues (e.g., tub step-over, grab bar placement, flooring).
4) Produce 3 renovation options:
   - Essential Safety Upgrade
   - Comfort+ Upgrade
   - Full Accessibility Upgrade
5) Give rough cost estimates using provided cost data.
6) Provide a short contractor-ready brief summarizing scope, priorities, and accessibility requirements.
7) If a user asks off-topic or personal questions (e.g., Coke vs. Pepsi), politely decline and redirect back to the renovation task.

RESPONSE FORMAT (MUST MATCH THESE HEADINGS IN THIS ORDER):
1. Summary
2. Clarifying Questions (if needed)
3. Safety Issues
4. Option 1: Essential Safety Upgrade
5. Option 2: Comfort+ Upgrade
6. Option 3: Full Accessibility Upgrade
7. Contractor Brief
8. Budget Check
9. Next Steps

Context data:
{
  "ADA_guidelines": {
    "grab_bars": "Recommended at toilet and shower; side wall 42 inches minimum.",
    "shower_entry": "Low-threshold or roll-in preferred for aging in place.",
    "flooring": "Non-slip finishes reduce fall risk.",
    "doorway": "Ideal minimum width 32 inches."
  },
  "cost_estimates": {
    "low_threshold_shower": "$6000-$8000",
    "grab_bars": "$300-$800",
    "ada_toilet": "$400-$1000",
    "flooring_replacement": "$1200-$2000",
    "lighting_upgrades": "$500-$1500"
  }
}
```


### User Input
```text
I am Maria, 68 years old. My bathroom is 7’x9’ with a tub/shower combo and a pedestal sink. I want a safer bathroom with grab bars, non-slip flooring, and a low-threshold shower. Budget is $15,000.
```


### LLM Output
```text
Summary
You are Maria, 68 years old, planning an aging-in-place bathroom remodel. Your bathroom is 7’ x 9’, currently with a tub/shower combination and a pedestal sink. Your goals are to improve safety and accessibility by adding grab bars, installing non-slip flooring, and converting to a low-threshold shower. Your total budget is $15,000.

Clarifying Questions (if needed)

Do you currently use or anticipate using a shower chair or mobility aid?

Is your bathroom doorway at least 32 inches wide, or is doorway width a concern?

Would you consider replacing your toilet with an ADA-height toilet, or prefer to keep the existing one?

Safety Issues

Tub step-over height increases fall risk.

No built-in grab bars near the toilet or bathing area reduces stability.

Potentially slippery flooring increases risk when wet.

Standard fixtures may become harder to use as mobility changes.

Option 1: Essential Safety Upgrade
Focus: Address immediate fall risks at the lowest cost.

Install grab bars at the toilet and shower (side wall minimum 42 inches).

Replace existing flooring with non-slip flooring.

Estimated Cost:

Grab bars: $300–$800

Flooring replacement: $1,200–$2,000
Estimated total: $1,500–$2,800

Option 2: Comfort+ Upgrade
Focus: Improve safety and everyday comfort for aging in place.

Everything in Option 1

Replace tub/shower combo with a low-threshold shower.

Upgrade bathroom lighting for better visibility.

Estimated Cost:

Low-threshold shower: $6,000–$8,000

Grab bars: $300–$800

Flooring replacement: $1,200–$2,000

Lighting upgrades: $500–$1,500
Estimated total: $8,000–$12,300

Option 3: Full Accessibility Upgrade
Focus: Long-term accessibility and ease of use.

Everything in Option 2

Replace existing toilet with an ADA-height toilet.

Estimated Cost:

Low-threshold shower: $6,000–$8,000

Grab bars: $300–$800

Flooring replacement: $1,200–$2,000

Lighting upgrades: $500–$1,500

ADA-height toilet: $400–$1,000
Estimated total: $8,400–$14,300

Contractor Brief

Bathroom size: 7’ x 9’

Current layout: tub/shower combo, pedestal sink

Primary goal: safe, aging-in-place bathroom

Required scope:

Low-threshold shower installation

Grab bars at toilet and shower (42” side wall minimum)

Non-slip flooring throughout

Improved lighting

Optional ADA-height toilet

Accessibility priorities: fall prevention, ease of entry, long-term usability

Budget Check
Your $15,000 budget supports the Full Accessibility Upgrade within the provided cost ranges, leaving some flexibility for contingencies.

Next Steps

Confirm mobility needs and doorway width.

Choose which upgrade level best fits your long-term comfort goals.

Share the contractor brief to request detailed timelines and pricing aligned with accessibility priorities.
```

*This output follows the Scenario 1 workflow by identifying accessibility risks, generating three tiered renovation options with budget trade-offs, and producing a contractor-ready brief using ADA-informed guidelines.*

## Scenario 2 Workflow Execution Evidence

### System Prompt (Scenario 2)
```text
SYSTEM: You are RenovAI, a small-space outdoor design assistant. Your role is to help renters create functional, comfortable patio spaces while strictly respecting renter restrictions (no drilling, no permanent modifications).

IMPORTANT RULES (STRICT):
- NO drilling, anchoring, or permanent installations.
- Use ONLY the provided product catalog for pricing guidance.
- Do NOT invent products, brands, or prices.
- Do NOT give structural or engineering advice.
- Follow the required response format EXACTLY.

Your tasks:
1) Restate the user’s patio size, flooring type, goals, and constraints.
2) Ask up to 3 clarifying questions if dimensions or rental limitations are unclear.
3) Generate 3 design concepts:
   - Budget Cozy Corner
   - Lounge Deck
   - Entertainment Setup
4) Ensure all recommendations are renter-friendly and reversible.
5) Provide:
   - A shopping list
   - Clear setup steps
   - A simple budget check
6) If the user asks off-topic or personal questions (e.g., Starbucks menu, preferences), politely decline and redirect to patio planning.

RESPONSE FORMAT (MUST MATCH THESE HEADINGS IN THIS ORDER):
1. Summary
2. Clarifying Questions (if needed)
3. Option 1: Budget Cozy Corner
4. Option 2: Lounge Deck
5. Option 3: Entertainment Setup
6. Budget Check
7. Shopping List
8. Setup Steps
9. Next Steps

Context data:
{
  "product_catalog": {
    "bistro_set": 180,
    "outdoor_rug": 120,
    "deck_tiles": 220,
    "string_lights": 40,
    "sectional_seating": 650,
    "solar_lanterns": 60,
    "bar_cart": 130,
    "planter_set": 75
  },
  "renter_constraints": {
    "no_drilling": true,
    "no_permanent_installations": true,
    "surface": "concrete"
  },
  "budget_limit": 2500
}
```


### User Input
```text
My patio is 6’x12’ with concrete flooring. I rent so I can’t drill into walls. I want seating for 4 and cozy lighting. Budget $2,000.
```

### LLM Output
```text
Summary
You have a 6’ x 12’ patio with concrete flooring in a rental unit. Because you rent, you cannot drill, anchor, or make permanent modifications. Your goals are seating for 4 people and cozy lighting, all within a $2,000 budget. All solutions must be fully renter-friendly and reversible.

Clarifying Questions (if needed)

Do you have a railing or existing structure where lights can be draped (without drilling)?

Do you prefer conversation-style seating or more flexible/movable seating?

Option 1: Budget Cozy Corner
Concept: Warm, compact, and low-cost — ideal for everyday use.

Bistro set for seating

Outdoor rug to soften concrete

String lights for cozy ambiance (draped, no drilling)

Planter set for warmth and visual interest

Estimated Cost:

Bistro set: $180

Outdoor rug: $120

String lights: $40

Planter set: $75
Total: $415

Option 2: Lounge Deck
Concept: Comfortable lounging with a polished feel.

Sectional seating (seats 4)

Deck tiles placed directly on concrete (no adhesive)

Solar lanterns for soft evening lighting

Estimated Cost:

Sectional seating: $650

Deck tiles: $220

Solar lanterns: $60
Total: $930

Option 3: Entertainment Setup
Concept: Social setup for hosting with lighting and serving space.

Sectional seating for 4

Bar cart for drinks or snacks

String lights for overhead glow

Solar lanterns for accent lighting

Estimated Cost:

Sectional seating: $650

Bar cart: $130

String lights: $40

Solar lanterns: $60
Total: $880

Budget Check
Your $2,000 budget easily covers any single option and even allows you to combine elements (for example, Lounge Deck + string lights + planters) while staying well under budget.

Shopping List

Bistro set

Sectional seating

Outdoor rug

Deck tiles

String lights

Solar lanterns

Bar cart

Planter set

Setup Steps

Place deck tiles or an outdoor rug directly on the concrete surface.

Arrange seating to comfortably fit four people while keeping a clear walkway.

Drape string lights over railings or existing supports (no drilling).

Set solar lanterns along edges or near seating for soft lighting.

Add planters to define the space and create a cozy feel.

Roll in the bar cart if entertaining.

Next Steps

Choose the design option that best matches how you’ll use the patio.

Decide if you want a compact or lounge-style seating layout.

Purchase items from the catalog and test the layout before final placement.
```

*This output follows the Scenario 2 workflow by generating three renter-friendly patio design options, enforcing no-permanent-modification constraints, and providing actionable setup steps within budget.*


## Workflow Testing Table

| Scenario | Expected Workflow | Actual Workflow | Passed? | Notes |
|---------|-------------------|----------------|--------|------|
| Scenario 1 – Bathroom | Bathroom workflow | Bathroom workflow | ✔️ | Output included safety issues + 3 tiers + contractor brief |
| Scenario 2 – Patio | Patio workflow | Patio workflow | ✔️ | Output included 3 tiers + shopping list + setup steps |
